In [23]:
import os
import sys
import subprocess

conda_path = "/opt/homebrew/Caskroom/miniconda/base/envs/proGAT"

In [24]:
dir_datas = "/Users/latterday/Desktop/Project/proGAT/datas"
dir_database = "/Users/latterday/Desktop/Project/proGAT/database"

In [25]:
file_temp = "/Users/latterday/Desktop/Project/proGAT/file_temp"
os.makedirs(file_temp, exist_ok=True)

In [26]:
if os.path.exists(dir_datas)==False:
    print(f"数据目录 {dir_datas} 不存在，请检查路径。")
    sys.exit(1)

In [27]:
sample_id = os.listdir(dir_datas)

In [28]:
sample_id = sample_id[1:]

seqkit 统计

In [29]:
temp_seqkit = os.path.join(file_temp, "seqkit_stats")
os.makedirs(temp_seqkit, exist_ok=True)

for sample in sample_id:
    temp_file = os.path.join(dir_datas, sample)
    sample_name = sample
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    subprocess.run(f"seqkit stats {temp_file}  -a -T > {temp_seqkit}/{sample_name}_stats.txt", shell=True)

fastplong 过滤

In [30]:
fastplong_bin = os.path.join(conda_path, "bin", "fastplong")

temp_fastplong = os.path.join(file_temp, "fastplong_filtered")
os.makedirs(temp_fastplong, exist_ok=True)

for sample in sample_id:
    input_file = os.path.join(dir_datas, sample)

    # 去除常见的 FASTQ 扩展名，得到干净的样本名称
    sample_name = sample
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break

    # 每个样本单独建立目录
    sample_output_dir = os.path.join(
        temp_fastplong,
        sample_name,
    )
    os.makedirs(sample_output_dir, exist_ok=True)

    output_fastq = os.path.join(
        sample_output_dir,
        f"{sample_name}_filtered.fastq.gz",
    )
    output_html = os.path.join(
        sample_output_dir,
        f"{sample_name}_fastplong.html",
    )
    output_json = os.path.join(
        sample_output_dir,
        f"{sample_name}_fastplong.json",
    )

    subprocess.run(
        [
            fastplong_bin,
            "-i", input_file,
            "-o", output_fastq,
            "-h", output_html,
            "-j", output_json,
        ],
        check=True,
    )

Trying to detect adapter sequence at read start
Not detected
Trying to detect adapter sequence at read end
Found possible adapter sequence, but it's too short: AGGTGCTGCAGGTA, specify -e AGGTGCTGCAGGTA to force trimming using this adapter

Before filtering:
total reads: 251505
total bases: 437387806
Q20 bases: 355258897(81.2229%)
Q30 bases: 284789508(65.1114%)

After filtering:
total reads: 241911
total bases: 419736197
Q20 bases: 347635869(82.8225%)
Q30 bases: 280672497(66.8688%)

Filtering result:
reads passed filter: 241911
reads failed due to low quality: 9594
reads failed due to too many N: 0
reads failed due to too short: 0
reads with adapter trimmed: 0
bases trimmed due to adapters: 0

JSON report: /Users/latterday/Desktop/Project/proGAT/file_temp/fastplong_filtered/SRR23100674/SRR23100674_fastplong.json
HTML report: /Users/latterday/Desktop/Project/proGAT/file_temp/fastplong_filtered/SRR23100674/SRR23100674_fastplong.html

/opt/homebrew/Caskroom/miniconda/base/envs/proGAT/bin/f

lrge 基因组大小预估

In [31]:
temp_lrge = os.path.join(file_temp, "lrge")
os.makedirs(temp_lrge, exist_ok=True)
docker_image_lrge = "staphb/lrge:latest"


for sample in sample_id:
    
    sample_name = os.path.basename(sample)
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    input_file = os.path.join(temp_fastplong, sample_name, f"{sample_name}_filtered.fastq.gz")
    
    output_name = f"{sample_name}_size.txt"
    
    subprocess.run(
        [
            "docker", "run", "--rm",
            "--platform", "linux/amd64",
            "-v", f"{input_file}:/input.fastq.gz:ro",
            "-v", f"{temp_lrge}:/output",
            docker_image_lrge,
            "lrge",
            "-P", "ont",
            "-t", "8",
            "-o", f"/output/{output_name}",
            "/input.fastq.gz",
        ],
        check=True,
    )
    

[2026-08-14T08:26:05Z INFO  lrge] Running two-set strategy with 10000 target reads and 5000 query reads
[2026-08-14T08:26:31Z INFO  liblrge::twoset] 313 (6.26%) query read(s) did not overlap any target reads
[2026-08-14T08:26:31Z INFO  lrge] Estimated genome size: 5.24 Mbp (IQR: 3.29 Mbp - 6.47 Mbp)
[2026-08-14T08:26:31Z INFO  lrge] Done!
[2026-08-14T08:26:31Z INFO  lrge] Running two-set strategy with 10000 target reads and 5000 query reads
[2026-08-14T08:26:39Z INFO  liblrge::twoset] 223 (4.46%) query read(s) did not overlap any target reads
[2026-08-14T08:26:39Z INFO  lrge] Estimated genome size: 5.51 Mbp (IQR: 3.30 Mbp - 7.09 Mbp)
[2026-08-14T08:26:39Z INFO  lrge] Done!


flye拼接

In [32]:
temp_flye = os.path.join(file_temp, "flye")
os.makedirs(temp_flye, exist_ok=True)

for sample in sample_id:
    sample_name = os.path.basename(sample)
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    input_file = os.path.join(temp_fastplong, sample_name, f"{sample_name}_filtered.fastq.gz")
    
    file_lrge = os.path.join(temp_lrge, f"{sample_name}_size.txt")
    
    subprocess.run([
        "flye",
        "--nano-hq", input_file,
        "--genome-size", "5m",
        "--out-dir", os.path.join(temp_flye, sample_name),
        "--threads", "8"
    ], check=True)


[2026-08-14 16:26:39] INFO: Starting Flye 2.9.6-b1802
[2026-08-14 16:26:39] INFO: >>>STAGE: configure
[2026-08-14 16:26:39] INFO: Configuring run
[2026-08-14 16:26:41] INFO: Total read length: 419736197
[2026-08-14 16:26:41] INFO: Input genome size: 5000000
[2026-08-14 16:26:41] INFO: Estimated coverage: 83
[2026-08-14 16:26:41] INFO: Reads N50/N90: 5003 / 584
[2026-08-14 16:26:41] INFO: Minimum overlap set to 1000
[2026-08-14 16:26:41] INFO: >>>STAGE: assembly
[2026-08-14 16:26:41] INFO: Assembling disjointigs
[2026-08-14 16:26:41] INFO: Reading sequences
[2026-08-14 16:26:44] INFO: Building minimizer index
[2026-08-14 16:26:44] INFO: Pre-calculating index storage
0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 
[2026-08-14 16:26:47] INFO: Filling index
0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 
[2026-08-14 16:26:52] INFO: Extending reads
[2026-08-14 16:27:04] INFO: Overlap-based coverage: 50
[2026-08-14 16:27:04] INFO: Median overlap divergence: 0.0336591
0% 80% 100% 
[2026-08-14 16:27

In [33]:
for sample in sample_id:
    sample_name = os.path.basename(sample)
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    subprocess.run([
        "seqkit", "stats",
        f"{temp_flye}/{sample_name}/assembly.fasta",
        "-a"])
    

file                                                                               format  type  num_seqs    sum_len  min_len    avg_len    max_len       Q1     Q2         Q3  sum_gap        N50  N50_num  Q20(%)  Q30(%)  AvgQual  GC(%)  sum_n
/Users/latterday/Desktop/Project/proGAT/file_temp/flye/SRR23100674/assembly.fasta  FASTA   DNA          3  4,974,504    2,688  1,658,168  4,967,151  3,676.5  4,665  2,485,908        0  4,967,151        1       0       0        0  50.61      0
file                                                                               format  type  num_seqs    sum_len  min_len    avg_len    max_len     Q1      Q2      Q3  sum_gap        N50  N50_num  Q20(%)  Q30(%)  AvgQual  GC(%)  sum_n
/Users/latterday/Desktop/Project/proGAT/file_temp/flye/SRR23100672/assembly.fasta  FASTA   DNA         24  5,216,710      172  217,362.9  3,266,462  2,423  13,269  95,379        0  3,266,462        1       0       0        0  50.81      0


QUAST 基本信息统计

In [34]:
temp_quast = os.path.join(file_temp, "quast")
os.makedirs(temp_quast, exist_ok=True)
docker_image_quast = "staphb/quast:latest"

for sample in sample_id:
    
    sample_name = os.path.basename(sample)
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    input_file = os.path.join(temp_flye, sample_name, "assembly.fasta")

    subprocess.run(
        [
            "docker", "run", "--rm",
            "--platform", "linux/amd64",
            "-v", f"{input_file}:/input.fasta:ro",
            "-v", f"{temp_quast}:/output",
            docker_image_quast,
            "quast.py",
            "/input.fasta",
            "-o", f"/output/{sample_name}_quast",
        ],
        check=True,
    )

/quast-5.3.0/quast.py /input.fasta -o /output/SRR23100674_quast

Version: 5.3.0

System information:
  OS: Linux-6.12.76-linuxkit-x86_64-with-glibc2.35 (linux_64)
  Python version: 3.10.12
  CPUs number: 10

Started: 2026-08-14 04:32:16

Logging to /output/SRR23100674_quast/quast.log
NOTICE: Output directory already exists and looks like a QUAST output dir. Existing results can be reused (e.g. previously generated alignments)!
NOTICE: Maximum number of threads is set to 2 (use --threads option to set it manually)

CWD: /data
Main parameters: 
  MODE: default, threads: 2, min contig length: 500, min alignment length: 65, min alignment IDY: 95.0, \
  ambiguity: one, min local misassembly length: 200, min extensive misassembly length: 1000

Contigs:
  Pre-processing...
  /input.fasta ==> input

2026-08-14 04:32:20
Running Basic statistics processor...
  Contig files: 
    input
  Calculating N50 and L50...
    input, N50 = 4967151, L50 = 1, auN = 4959814.7, Total length = 4974504, GC % = 

CheckM2 完整度+污染率 评估

In [35]:
temp_checkm = os.path.join(file_temp, "checkm")
os.makedirs(temp_checkm, exist_ok=True)
docker_image_checkm = "staphb/checkm2:latest"
database_checkm = os.path.join(dir_database, "checkm2")
database_file = os.path.join(
    database_checkm,
    "CheckM2_database",
    "uniref100.KO.1.dmnd",
)


for sample in sample_id:
    
    sample_name = os.path.basename(sample)
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    input_file = os.path.join(temp_flye, sample_name, "assembly.fasta")
    
    threads = 8
    subprocess.run([
        "docker", "run", "--rm",
        "--platform", "linux/amd64",
        "-v", f"{input_file}:/input.fasta:ro",
        "-v", f"{temp_checkm}:/output",
        "-v", f"{database_file}:/checkm_data.dmnd:ro",
        docker_image_checkm,
        "checkm2", "predict",
        "--threads", str(threads),
        "--input", "/input.fasta",
        "--output-directory", f"/output/{sample_name}_checkm2",
        "--database", "/checkm_data.dmnd",
    ], check=True)



[08/14/2026 08:32:57 AM] INFO: Running CheckM2 version 1.1.0
[08/14/2026 08:32:57 AM] INFO: Custom database path provided for predict run. Checking database at /checkm_data.dmnd...
[08/14/2026 08:33:34 AM] INFO: Running quality prediction workflow with 8 threads.
[08/14/2026 08:33:35 AM] INFO: Calling genes in 1 bins with 8 threads:


    Finished processing 1 of 1 (100.00%) bins.


[08/14/2026 08:35:17 AM] INFO: Calculating metadata for 1 bins with 8 threads:


    Finished processing 1 of 1 (100.00%) bin metadata.


[08/14/2026 08:35:18 AM] INFO: Annotating input genomes with DIAMOND using 8 threads
[08/14/2026 08:41:16 AM] INFO: Processing DIAMOND output
[08/14/2026 08:41:17 AM] INFO: Predicting completeness and contamination using ML models.
[08/14/2026 08:41:39 AM] INFO: Parsing all results and constructing final output table.
[08/14/2026 08:41:39 AM] INFO: CheckM2 finished successfully.
[08/14/2026 08:41:52 AM] INFO: Running CheckM2 version 1.1.0
[08/14/2026 08:41:52 AM] INFO: Custom database path provided for predict run. Checking database at /checkm_data.dmnd...
[08/14/2026 08:42:21 AM] INFO: Running quality prediction workflow with 8 threads.
[08/14/2026 08:42:22 AM] INFO: Calling genes in 1 bins with 8 threads:


    Finished processing 1 of 1 (100.00%) bins.


[08/14/2026 08:44:03 AM] INFO: Calculating metadata for 1 bins with 8 threads:


    Finished processing 1 of 1 (100.00%) bin metadata.


[08/14/2026 08:44:04 AM] INFO: Annotating input genomes with DIAMOND using 8 threads
[08/14/2026 08:49:31 AM] INFO: Processing DIAMOND output
[08/14/2026 08:49:32 AM] INFO: Predicting completeness and contamination using ML models.
[08/14/2026 08:49:52 AM] INFO: Parsing all results and constructing final output table.
[08/14/2026 08:49:52 AM] INFO: CheckM2 finished successfully.


merqury 检查碱基准确性和序列完整性

In [36]:
# temp_merquy = os.path.join(file_temp, "merqury")
# os.makedirs(temp_merquy, exist_ok=True)
# docker_image_merqury = "danylmb/merqury:1.4.1-build2"


# for sample in sample_id:
    
#     sample_name = os.path.basename(sample)
#     for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
#         if sample_name.endswith(suffix):
#             sample_name = sample_name.removesuffix(suffix)
#             break
    
#     input_file_merqury = os.path.join(temp_flye, sample_name, "assembly.fasta")
#     input_file_meryl = os.path.join(temp_fastplong, sample_name, f"{sample_name}_filtered.fastq.gz")
    
    
#     temp_path = os.path.join(temp_merquy, sample_name)
#     os.makedirs(temp_path, exist_ok=True)
    
#     read_meryl = os.path.join(temp_path, "meryl_dir")
#     os.makedirs(read_meryl, exist_ok=True)
    
#     subprocess.run([
#     "docker", "run", "--rm",
#     "-v", f"{input_file_meryl}:/input.fastq.gz:ro",
#     "-v", f"{read_meryl}:/output",
#     docker_image_merqury,
#     "meryl",
#     "count",
#     "k=17",
#     "output", 
#     "/output/read.meryl",
#     "/input.fastq.gz",
#     ], check=True)
    
    
#     subprocess.run([
#         "docker", "run", "--rm",
#         "-v", f"{read_meryl}:/input_meryl:ro",
#         "-v", f"{input_file_merqury}:/input.fasta:ro",
#         docker_image_merqury,
#         "merqury.sh",
#         f"/input_meryl/input.meryl",
#         f"/input.fasta",
#         f"merqury_res"
#     ],check=True)

In [42]:
busco_db

'/Users/latterday/Desktop/Project/proGAT/database/busco'

In [ ]:
# BUSCO 模型下载
busco_db = os.path.join(dir_database, "busco")
os.makedirs(busco_db, exist_ok=True)

# 模型list(推荐terminal中运行)
subprocess.run([
    "docker", "run", "--rm",
    "--platform", "linux/amd64",
    "-v", f"{busco_db}:/busco_db",
    "staphb/busco:latest",
    "busco", "--list-datasets"
], check=True)
# docker run --rm -it --platform linux/amd64 staphb/busco:latest busco --list-datasets

2026-08-17 01:40:34 INFO:	Downloading information on latest versions of BUSCO data...
2026-08-17 01:40:42 INFO:	Downloading file 'https://busco-data.ezlab.org/v5/data/information/lineages_list.2026-05-26.txt.tar.gz'
2026-08-17 01:40:45 INFO:	Decompressing file '/data/busco_downloads/information/lineages_list.2026-05-26.txt.tar.gz'
################################################

Datasets available to be used with BUSCO v6.1.0 and later (numbers in brackets indicate the number of marker gene profiles):

- archaea_odb12.2 [195]
    - euryarchaeota_odb12.2 [282]
        - methanomicrobia_odb12.2 [589]
            - methanosarcinaceae_odb12.2 [971]
            - methanosarcina_odb12.2 [1620]
            - methanomicrobiales_odb12.2 [819]
               - methanomicrobiaceae_odb12.2 [993]
        - halobacteria_odb12.2 [807]
            - halobacteriales_odb12.2 [909]
                - halobacteriaceae_odb12.2 [975]
                - haloarculaceae_odb12.2 [1058]
                    - halo

CompletedProcess(args=['docker', 'run', '--rm', '--platform', 'linux/amd64', '-v', '/Users/latterday/Desktop/Project/proGAT/database/busco:/busco_db', 'staphb/busco:latest', 'busco', '--list-datasets'], returncode=0)

In [45]:
subprocess.run([
    "docker", "run", "--rm",
    "--platform", "linux/amd64",
    "-v", f"{busco_db}:/busco_db",
    "staphb/busco:latest",
    "busco", "--download", "bacteria_odb12", 
    "--download_path", "/busco_db/bacteria_odb12"
], check=True)

2026-08-17 01:41:53 INFO:	Downloading information on latest versions of BUSCO data...
2026-08-17 01:42:03 INFO:	Downloading file 'https://busco-data.ezlab.org/v5/data/lineages/bacteria_odb12.2026-05-22.tar.gz'
2026-08-17 01:43:32 INFO:	Decompressing file '/busco_db/bacteria_odb12/lineages/bacteria_odb12.tar.gz'


CompletedProcess(args=['docker', 'run', '--rm', '--platform', 'linux/amd64', '-v', '/Users/latterday/Desktop/Project/proGAT/database/busco:/busco_db', 'staphb/busco:latest', 'busco', '--download', 'bacteria_odb12', '--download_path', '/busco_db/bacteria_odb12'], returncode=0)

In [ ]:
temp_busco = os.path.join(file_temp, "BUSCO")
os.makedirs(temp_busco, exist_ok=True)
docker_image_busco = "staphb/busco:latest"
busco_db_temp = os.path.join(busco_db, "bacteria_odb12","lineages")

for sample in sample_id:
    
    sample_name = os.path.basename(sample)
    for suffix in (".fastq.gz", ".fq.gz", ".fastq", ".fq"):
        if sample_name.endswith(suffix):
            sample_name = sample_name.removesuffix(suffix)
            break
    
    input_file = os.path.join(temp_flye, sample_name, "assembly.fasta")
    output_dir = os.path.join(temp_busco, sample_name)
    os.makedirs(os.path.join(output_dir), exist_ok=True)
    
    
    subprocess.run([
        "docker", "run", "--rm",
        "--platform", "linux/amd64",
        "-v", f"{input_file}:/input.fasta:ro",
        "-v", f"{output_dir}:/output",
        "-v", f"{busco_db_temp}:/busco_db:ro",
        docker_image_busco,
        "busco",
        "-i", "/input.fasta",
        "-o", "/busco_result",
        "--out_path", "/output",
        "-l", "/busco_db/bacteria_odb12",
        "-m", "genome",
        "--cpu", "8",
        "--offline","-f"
    ])
    
    